## Meno vs. DeepONet
**Computational speedup**

Import libraries

In [1]:
import os
import pandas as pd
import tensorflow as tf

In [2]:
import utils

Define inputs

In [3]:
# "csv" header
header = ["t", "mean", "std"]
# Paths to timings
prefix = "./core/"
# Final times
tend = [1e-8, 1e-7, 1e-6, 1e-5, 1e-4]
# Timings
models = {
  "num": "meno_f64",
  "den": "deeponet_r06"
}
models_name = {
  "meno_f64": "MENO",
  "deeponet_r06": "DeepONet"
}
devices = ("cpu", "gpu")

In [4]:
path_to_saving = os.path.abspath("./../output/speedup/deeponet/")
os.makedirs(path_to_saving, exist_ok=True)

Read timings

In [5]:
def read_timings(filename, header, tend=None):
  timings = pd.read_csv(filename)
  timings.columns = header
  if (tend is not None):
    timings = timings[timings['t'].isin(tend)]
  return timings

In [6]:
timingsf = {}
for model in models.values():
  timingsf[model] = {}
  for device in devices:
    filename = prefix + f"/sur/data/timings_{model}_{device}.csv"
    timingsf[model][device] = filename

In [7]:
timings = tf.nest.map_structure(lambda f: read_timings(f, header, tend), timingsf)

Compute speedup

In [8]:
speedup = {}
for device in devices:
  num = timings[models["num"]][device]
  den = timings[models["den"]][device]
  speedup[device] = utils.compute_speedup(num, den, header)

In [9]:
speedup_flat = pd.json_normalize(speedup, sep="_").to_dict(orient="records")[0]

In [10]:
ipath = path_to_saving + "/data/"
os.makedirs(ipath, exist_ok=True)
for (ispeedup, idata) in speedup_flat.items():
  utils.save_speedup(
    df=idata,
    filename=ipath + ispeedup
  )

Plot timings

In [11]:
ipath = path_to_saving + "/figs/"
os.makedirs(ipath, exist_ok=True)

In [12]:
for idata in speedup_flat.values():
  time = idata["t"].to_numpy().squeeze()
  break

> Speedup - 1D plot

In [13]:
for device in devices:
  idata = {"x": time, "mean": None, "std": None}
  for k in ("mean", "std"):
    idata[k] = [speedup[device][k].values]
  utils.plot_timings_1d(
    **idata,
    var_names=None,
    labels=["$t_f$ [s]", "Speedup"],
    scales=["log", "linear"],
    figname=ipath + f"/speedup_{device}",
    save=True,
    show=False,
    extensions=("pdf",)
  )

> Computational Time - 1D plot

In [14]:
for device in devices:
  idata = {
    "x": timings[model][device]["t"].values,
    "mean": [timings[model][device]["mean"].values for model in models.values()],
    "std": None
  }
  utils.plot_timings_1d(
    **idata,
    var_names=[models_name[model] for model in models.values()],
    labels=["$t_f$ [s]", "Computational Time [s]"],
    scales=["log", "linear"],
    figname=ipath + f"/comp_time_{device}",
    save=True,
    show=False,
    extensions=("pdf",)
  )